# Signal shape — performance signals (X)

*Read-only informative artifact. Characterises each candidate signal so a
human can decide which carry usable information. No gate decisions, no
PROCEED/STOP verdict.*

## Questions a manager (and an analyst) asks of the signals

- **For each signal, at each position, what does its distribution look like?**
  Where is its mass, how much does it vary, how much of it is just zero?
- **Which signals are effectively dead at a position?** A signal with almost no
  variance, or that is zero almost all the time, cannot separate players and
  carries little usable information there.
- A goalkeeper has no `xg`; a defender's `threat` is mostly zero. Which
  signal/position pairs are structurally degenerate?

Everything below is **season-pooled** over the study range. How a signal
evolves week-to-week is deferred to the `temporal/` layer.

## Setup

Load the mart, restrict to the **whole season** (GW 1 to the latest completed
GW) and the **participation** population (`minutes > 0`), build position
cohorts, and identify the candidate signal columns.

This is a *descriptive characterisation* notebook, so it uses the full season —
no early-GW lower bound (that GW-6 cut in the older EDA-1 record was a
*predictive-evaluation* choice, not relevant here).

The population is everyone who **actually featured**: available players with
`minutes > 0`. This is a **participation** filter (the player appeared), **not
a performance gate**. `minutes` can be NULL for some rows; `minutes > 0`
naturally excludes those. The 60-minute performance-boundary question is
**deferred to the `population/` layer** — the layer meant to study and justify
that boundary — and is deliberately not baked in here.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.kernels.descriptive.distribution import (
    compute_distribution_stats,
    compare_cohorts,
)

try:
    _result = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _result = load_mart()

# Descriptive characterisation uses the WHOLE season: GW 1 to the latest
# completed GW. No early-GW lower bound (that was a predictive-evaluation
# choice in the older EDA-1 record, not relevant here).
STUDY_GW_MIN = 1
STUDY_GW_MAX = _result.data_cutoff_gw

# Analytical population: PARTICIPATION filter, not a performance gate.
# Available players who actually featured -> minutes > 0. `minutes` can be NULL
# for some rows; minutes > 0 naturally excludes NULLs (NULL comparisons are
# False), stated explicitly below. The 60-minute performance boundary is NOT
# imposed here -- that question is deferred to the population/ layer.
mart = _result.mart
df = mart[mart["gw"].between(STUDY_GW_MIN, STUDY_GW_MAX)].copy()
df = df[df["minutes"].notna() & (df["minutes"] > 0)].copy()

POSITIONS = ["GK", "DEF", "MID", "FWD"]
cohorts = {pos: df[df.position == pos] for pos in POSITIONS}

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.3f}".format)

print(f"Study range: GW {STUDY_GW_MIN} - GW {STUDY_GW_MAX} (whole season, from mart data_cutoff_gw)")
print(f"Population: minutes > 0 (participation, not a performance gate), n = {len(df):,} player-gameweeks")
for pos in POSITIONS:
    print(f"  {pos}: {len(cohorts[pos]):>6,}")

# Candidate signals: numeric per-gameweek performance signals carried by the
# mart (read from dal/mart/mart_schema.py + the live mart). Rolling/contextual
# columns and identity/market/structural columns are excluded — those are not
# raw weekly performance signals.
SIGNALS = [
    "xg", "xa", "xgi", "xgc",
    "ict_index", "influence", "creativity", "threat",
    "bps", "bonus", "shots" if "shots" in df.columns else "saves",
]
SIGNALS = [s for s in SIGNALS if s in df.columns]
print("\nCandidate signals:", SIGNALS)


## (a) Distribution, variance and zero-mass by position, per signal

**What we measure** — for every (signal, position) pair: the full distribution
summary from `compute_distribution_stats` (mean, std, variance, percentiles,
skew) plus the **zero-mass** fraction (`share of rows == 0`), assembled into
one long table.

**What it means** — this is the per-position fingerprint of each signal. The
**variance** says whether a signal separates players at that position at all;
the **zero-mass** says how often the signal simply does not fire (a striker's
`saves`, a keeper's `xg`). High variance with moderate zero-mass is the
profile of an informative signal; near-zero variance or near-total zero-mass
is the profile of a dead one.

**What it doesn't mean** — these are **season-pooled** moments. A signal can
have healthy pooled variance yet be flat within a single player's season
(between-player variance, not within-player) — that distinction is deferred to
the `temporal/` layer. Variance here measures *spread*, not *predictive value*
for `total_points`; association with the target is out of scope for this
layer.

**Guiding question** — *For each signal at each position, where is its mass,
how much does it vary, and how often is it just zero?*

In [ ]:
rows = []
for sig in SIGNALS:
    for pos in POSITIONS:
        s = cohorts[pos][sig].dropna().astype(float)
        stats = compute_distribution_stats(s)
        zero_mass = float((s == 0).mean()) if len(s) else np.nan
        rows.append({
            "signal": sig,
            "position": pos,
            "n": int(stats["count"]) if not np.isnan(stats["count"]) else 0,
            "mean": stats["mean"],
            "std": stats["std"],
            "variance": stats["variance"],
            "p90": stats["p90"],
            "max": stats["max"],
            "skew": stats["skew"],
            "zero_mass_%": round(zero_mass * 100, 1) if not np.isnan(zero_mass) else np.nan,
        })
signal_profile = pd.DataFrame(rows)
display(signal_profile.round(3))


In [ ]:
# A heatmap reveals shape the long table hides: which signal/position cells
# are dead (high zero-mass) at a glance.
pivot = signal_profile.pivot(index="signal", columns="position", values="zero_mass_%")
pivot = pivot[POSITIONS]
fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(pivot.values, cmap="magma_r", aspect="auto", vmin=0, vmax=100)
ax.set_xticks(range(len(POSITIONS)))
ax.set_xticklabels(POSITIONS)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center",
                    color="white" if v > 50 else "black", fontsize=8)
ax.set_title("Zero-mass % by signal x position")
fig.colorbar(im, ax=ax, label="% of rows == 0")
plt.tight_layout()
plt.show()


## (b) Degenerate-signal flag

**What we measure** — a flag on each (signal, position) pair that is
effectively dead, using two **operational heuristics** (stated, not derived):

- `near_zero_variance` — coefficient of variation is undefined or the
  **variance is below 1e-4** (the signal barely moves at this position);
- `high_zero_mass` — **zero-mass >= 90%** (the signal almost never fires here).

A pair is flagged `degenerate` if either condition holds. These thresholds are
heuristics for triage, not statistical tests.

**What it means** — a flagged pair carries little usable information at that
position and is a candidate to drop from position-specific analysis (a keeper's
`xg`, a defender's `saves`). It directs attention; it is not a verdict.

**What it doesn't mean** — "degenerate" means *low spread / mostly-zero in this
pooled population*, not "useless". A rare-but-decisive signal (fires 5% of
weeks but those weeks haul) would be flagged `high_zero_mass` yet still matter;
the flag does not test association with `total_points`. The 1e-4 and 90% cut
points are arbitrary operational lines. All **season-pooled**.

**Guiding question** — *Which signal/position pairs are effectively dead and
should be set aside?*

In [ ]:
NEAR_ZERO_VARIANCE = 1e-4   # operational heuristic: variance below this == flat
HIGH_ZERO_MASS_PCT = 90.0    # operational heuristic: >= this % zeros == rarely fires

flagged = signal_profile.copy()
flagged["near_zero_variance"] = flagged["variance"] < NEAR_ZERO_VARIANCE
flagged["high_zero_mass"] = flagged["zero_mass_%"] >= HIGH_ZERO_MASS_PCT
flagged["degenerate"] = flagged["near_zero_variance"] | flagged["high_zero_mass"]

print(f"Heuristics: variance < {NEAR_ZERO_VARIANCE:g}  OR  zero_mass >= {HIGH_ZERO_MASS_PCT}%")
print(f"Flagged degenerate: {int(flagged['degenerate'].sum())} of {len(flagged)} (signal, position) pairs\n")
display(
    flagged.loc[
        flagged["degenerate"],
        ["signal", "position", "variance", "zero_mass_%", "near_zero_variance", "high_zero_mass"],
    ].reset_index(drop=True)
)


## What the signals look like

Plain-language summary (not a verdict):

- Attacking signals (`xg`, `xa`, `xgi`, `threat`, `creativity`) are heavily
  **zero-massed for GK and DEF** and only carry real spread for MID and FWD —
  defenders rarely generate attacking output.
- `xgc` and (where present) `saves` are the mirror image: alive for GK/DEF,
  near-dead for FWD.
- Composite signals (`ict_index`, `bps`, `influence`) carry spread across all
  positions because they aggregate multiple contributions.
- The degenerate flag concentrates exactly where football structure predicts:
  attacking metrics at the back, defensive metrics up front.

All **whole-season**, season-pooled, over the **participation** population
(`minutes > 0` — the player appeared — not a performance gate; the 60-minute
boundary is deferred to the `population/` layer). Within-player signal
behaviour is deferred to the `temporal/` layer.